# Task 5: Exploratory Data Analysis (EDA) — Titanic Dataset

**Objective:** Extract insights using visual and statistical exploration.

**Tools:** Python, Pandas, Matplotlib, Seaborn.

This notebook downloads the commonly used Titanic training dataset from a public GitHub CSV when you run it. Internet access is required for the first data-loading step. The notebook then performs data inspection, cleaning, univariate/bivariate/multivariate analysis, visualizations, and a summary.

## 1. Import libraries and load the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

DATA_URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(DATA_URL)
print('Dataset shape:', df.shape)
df.head()

## 2. Understand the data

Use `head()`, `tail()`, `shape`, `info()`, `describe()` and `value_counts()` to understand the dataset.

In [ ]:
display(df.head())
print('Shape:', df.shape)
print('\nColumn information:')
df.info()
print('\nNumerical summary:')
display(df.describe(include='all').T)

if 'Survived' in df.columns:
    print('\nSurvival counts:')
    display(df['Survived'].value_counts(dropna=False))
    print('\nSurvival percentages:')
    display((df['Survived'].value_counts(normalize=True) * 100).round(2))

## 3. Data quality checks and cleaning

Check missing values and duplicates. For this introductory analysis, missing age values are filled with the median, missing embarkation values with the mode, and missing cabin values are retained as a missingness indicator. We create a separate working copy so the original data remains unchanged.

In [ ]:
print('Missing values by column:')
display(df.isna().sum().sort_values(ascending=False))
print('Duplicate rows:', df.duplicated().sum())

eda = df.copy()
if 'Age' in eda.columns:
    eda['Age'] = eda['Age'].fillna(eda['Age'].median())
if 'Embarked' in eda.columns:
    mode_embarked = eda['Embarked'].mode()
    if not mode_embarked.empty:
        eda['Embarked'] = eda['Embarked'].fillna(mode_embarked.iloc[0])
if 'Cabin' in eda.columns:
    eda['HasCabin'] = eda['Cabin'].notna().astype(int)

print('Missing values after selected cleaning:')
display(eda.isna().sum().sort_values(ascending=False))

## 4. Univariate analysis

Histograms show the distribution of numerical variables. Count plots show the frequency of categorical variables.

In [ ]:
numeric_cols = [c for c in ['Age', 'Fare', 'SibSp', 'Parch'] if c in eda.columns]
for col in numeric_cols:
    plt.figure()
    sns.histplot(data=eda, x=col, kde=True)
    plt.title(f'Distribution of {col}')
    plt.tight_layout()
    plt.show()

for col in [c for c in ['Sex', 'Pclass', 'Embarked', 'Survived'] if c in eda.columns]:
    plt.figure()
    sns.countplot(data=eda, x=col)
    plt.title(f'Count of {col}')
    plt.tight_layout()
    plt.show()

**Observation guide:** Note whether numerical distributions are symmetric or skewed, whether outliers are present, and which categories occur most frequently. In the standard Titanic training data, fares are strongly right-skewed and third-class passengers form the largest passenger class.

## 5. Bivariate analysis

Compare survival with passenger class, sex, and age. Boxplots help compare numerical distributions across groups.

In [ ]:
if {'Survived', 'Sex'}.issubset(eda.columns):
    display(pd.crosstab(eda['Sex'], eda['Survived'], normalize='index').round(3))
    sns.barplot(data=eda, x='Sex', y='Survived', errorbar=None)
    plt.title('Survival rate by sex')
    plt.ylabel('Survival rate')
    plt.show()

if {'Survived', 'Pclass'}.issubset(eda.columns):
    display(pd.crosstab(eda['Pclass'], eda['Survived'], normalize='index').round(3))
    sns.barplot(data=eda, x='Pclass', y='Survived', errorbar=None)
    plt.title('Survival rate by passenger class')
    plt.ylabel('Survival rate')
    plt.show()

if {'Survived', 'Age'}.issubset(eda.columns):
    sns.boxplot(data=eda, x='Survived', y='Age')
    plt.title('Age distribution by survival status')
    plt.show()

**Observation guide:** Compare the heights of the survival-rate bars. In the commonly used Titanic training dataset, women and first-class passengers had higher survival rates than men and third-class passengers. These are associations in this dataset, not proof of causation.

## 6. Multivariate analysis and correlation

A correlation heatmap summarizes linear relationships among numeric columns. A pairplot shows pairwise patterns for a small set of variables.

In [ ]:
numeric_data = eda.select_dtypes(include=np.number)
plt.figure(figsize=(10, 7))
sns.heatmap(numeric_data.corr(), annot=True, cmap='coolwarm', fmt='.2f', center=0)
plt.title('Correlation heatmap')
plt.tight_layout()
plt.show()

pair_cols = [c for c in ['Survived', 'Pclass', 'Age', 'Fare', 'SibSp'] if c in eda.columns]
sns.pairplot(eda[pair_cols].dropna().sample(min(300, len(eda)), random_state=42), corner=True)
plt.show()

## 7. Outlier detection with boxplots

In [ ]:
for col in [c for c in ['Age', 'Fare'] if c in eda.columns]:
    plt.figure()
    sns.boxplot(x=eda[col])
    plt.title(f'Boxplot of {col}')
    plt.show()

# IQR-based outlier counts (not automatically removed)
for col in [c for c in ['Age', 'Fare'] if c in eda.columns]:
    q1, q3 = eda[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((eda[col] < lower) | (eda[col] > upper)).sum()
    print(f'{col}: {count} potential IQR outliers')

## 8. Key findings and conclusion

After running the notebook, use the outputs above to write dataset-specific observations. A concise conclusion should mention: (1) the overall survival distribution, (2) differences by sex and passenger class, (3) skewness/outliers in fare, (4) missing data and the cleaning decisions, and (5) notable correlations. Remember that correlation does not imply causation.

### Exporting

Use **File → Save and Export Notebook As → HTML/PDF** in Jupyter if your submission requires a rendered notebook. The separate PDF report included with this package is a written report template with high-level findings for the standard Titanic training dataset.